# deep_capitulation_reentry_guard_v1

Purpose: test whether the stable deep-capitulation label from `stable_rebound_label_discovery_v1` can be used as a constrained execution re-entry floor, without retraining PPO and without reopening direct SOXX/SOXL spread features.

Governance:
- Baseline: `v19reg_v22_budget_usage_bull_guard_v1` and the stricter `recovery_reentry_equity_drawdown_guard_v1` drawdown-control line.
- Direct comparison: bull-participation control and equity-drawdown control on the same fixed slow/fast actors.
- Role: `diagnose`; this is an execution/feature-target handoff check, not a promotion candidate by default.
- Changed layer: `execution` with a default-off guard in the shared two-speed composer.
- Target window: strict OOS `2020-01-02..2026-05-21`, then saved walk-forward checks for `wf_2020`, `wf_2022`, and `wf_2023_2024`.
- Accepted trade-off: preserve at least `+30pp` strict-OOS excess versus buy-and-hold while reducing path risk; hard objective remains MaxDD `<=15%`.
- Failure mode: drawdown remains far above `15%`, walk-forward regressions appear, or the guard only adds churn without order-generation impact.
- Comparison-table row key: `deep_capitulation_reentry_guard_v1`.

External basis:
- Qlib reproducible workflow and fixed-harness evaluation: `https://github.com/microsoft/qlib`, `https://arxiv.org/abs/2009.11189`.
- Time-series momentum position-sizing separation: `https://arxiv.org/abs/1904.04912`.
- Backtest-overfitting discipline for small parameter grids and walk-forward checks: `https://doi.org/10.21314/JCF.2016.322`.
- Drawdown as path-dependent risk, not only a terminal statistic: `https://papers.ssrn.com/sol3/papers.cfm?abstract_id=544742`.

Historical reflection:
- Closest success: the retained selective-release fixed-actor overlay improved recovery/proxy slices, but prior evidence showed eligible days can be absorbed by execution thresholds.
- Closest failure: direct SOXX/SOXL spread retraining and fold-local state labels produced unstable or non-promotable signals.
- Difference from both: this loop uses a sparse feature-defined label only to floor low targets in deep capitulation states, then checks actual order-generation and walk-forward behavior before any training handoff.


In [1]:
from __future__ import annotations

import json
import subprocess
import sys
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import Markdown, display


def find_repo_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'AGENTS.md').exists() and (candidate / 'src' / 'soxl_quant').exists():
            return candidate
    raise RuntimeError('repo root not found')


REPO_ROOT = find_repo_root(Path.cwd())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.soxl_quant.app.budget_counterfactual import (
    BullParticipationGuardConfig,
    DeepCapitulationReentryGuardConfig,
    EquityDrawdownControlConfig,
    align_price,
    build_budget_usage_counterfactual,
    compute_metrics,
    dashboard_daily_frame,
)
from src.soxl_quant.data import build_dataset, init_qlib
from src.soxl_quant.policy.two_speed_budget import TwoSpeedBudgetConfig
from src.soxl_quant.settings import QLIB_DATA_PATH

EXPERIMENT = 'deep_capitulation_reentry_guard_v1'
OUT_DIR = REPO_ROOT / 'runtime' / 'execution' / 'card' / EXPERIMENT
INPUT_DIR = OUT_DIR / 'inputs'
OUT_DIR.mkdir(parents=True, exist_ok=True)
INPUT_DIR.mkdir(parents=True, exist_ok=True)

SLOW_PROFILE = 'balanced_rule_residual_v19_reg'
FAST_PROFILE = 'balanced_rule_residual_v22'
SLOW_CURRENT = REPO_ROOT / 'runtime' / 'model' / 'card' / SLOW_PROFILE / f'{SLOW_PROFILE}_soxl_daily.csv'
FAST_CURRENT = REPO_ROOT / 'runtime' / 'model' / 'card' / FAST_PROFILE / f'{FAST_PROFILE}_soxl_daily.csv'

print({'repo_root': str(REPO_ROOT), 'out_dir': str(OUT_DIR), 'slow_current': str(SLOW_CURRENT), 'fast_current': str(FAST_CURRENT)})


{'repo_root': '/Users/rziiiii/.openclaw/workspace', 'out_dir': '/Users/rziiiii/.openclaw/workspace/runtime/execution/card/deep_capitulation_reentry_guard_v1', 'slow_current': '/Users/rziiiii/.openclaw/workspace/runtime/model/card/balanced_rule_residual_v19_reg/balanced_rule_residual_v19_reg_soxl_daily.csv', 'fast_current': '/Users/rziiiii/.openclaw/workspace/runtime/model/card/balanced_rule_residual_v22/balanced_rule_residual_v22_soxl_daily.csv'}


In [2]:
init_qlib(QLIB_DATA_PATH)
dataset = build_dataset()

price_frame = dataset.handler.price_frame.copy()
if isinstance(price_frame.index, pd.MultiIndex):
    price = price_frame.droplevel('instrument')['price'].sort_index()
else:
    price = price_frame['price'].sort_index()
price.index = pd.to_datetime(price.index)
price = price.astype(float)

feature_cols = [
    'mom_5',
    'drawdown_63',
    'mom_10',
    'mom_21',
    'relative_strength_21',
    'vol_of_vol_21',
    'vix_percentile_252',
    'anchor_gap_180',
    'below_anchor_ratio_63',
]
features = dataset.handler.feature_frame[feature_cols].copy()
features = features.reset_index()
if 'instrument' in features.columns:
    features = features.drop(columns=['instrument'])
features['datetime'] = pd.to_datetime(features['datetime'])
features = features.sort_values('datetime').reset_index(drop=True)


def read_daily(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, parse_dates=['datetime']).sort_values('datetime').reset_index(drop=True)
    if 'position' not in frame.columns:
        raise ValueError(f'{path} missing position column')
    return frame[['datetime', 'position']].copy()

slow_current = read_daily(SLOW_CURRENT)
fast_current = read_daily(FAST_CURRENT)

input_manifest = {
    'price_start': str(price.index.min().date()),
    'price_end': str(price.index.max().date()),
    'feature_start': str(features['datetime'].min().date()),
    'feature_end': str(features['datetime'].max().date()),
    'slow_current': str(SLOW_CURRENT.relative_to(REPO_ROOT)),
    'fast_current': str(FAST_CURRENT.relative_to(REPO_ROOT)),
    'slow_rows': int(len(slow_current)),
    'fast_rows': int(len(fast_current)),
}
(OUT_DIR / 'input_manifest.json').write_text(json.dumps(input_manifest, indent=2), encoding='utf-8')
input_manifest


[42154:MainThread](2026-05-25 23:45:03,895) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[42154:MainThread](2026-05-25 23:45:04,574) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[42154:MainThread](2026-05-25 23:45:04,575) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/.openclaw/workspace/data/qlib/soxl')}


[42154:MainThread](2026-05-25 23:45:07,308) INFO - qlib.timer - [log.py:127] - Time cost: 0.001s | Loading data Done


[42154:MainThread](2026-05-25 23:45:07,310) INFO - qlib.timer - [log.py:127] - Time cost: 0.000s | fit & process data Done


[42154:MainThread](2026-05-25 23:45:07,314) INFO - qlib.timer - [log.py:127] - Time cost: 0.007s | Init data Done


{'price_start': '2011-03-11',
 'price_end': '2026-05-22',
 'feature_start': '2011-03-11',
 'feature_end': '2026-05-22',
 'slow_current': 'runtime/model/card/balanced_rule_residual_v19_reg/balanced_rule_residual_v19_reg_soxl_daily.csv',
 'fast_current': 'runtime/model/card/balanced_rule_residual_v22/balanced_rule_residual_v22_soxl_daily.csv',
 'slow_rows': 1605,
 'fast_rows': 1605}

In [3]:
WINDOWS = {
    'test_2020': ('2020-01-02', '2020-12-31'),
    'test_2022': ('2022-01-03', '2022-12-30'),
    'test_2023_2024': ('2023-01-03', '2024-12-31'),
}
WF_WINDOWS = {
    'wf_2020': ('2020-01-02', '2020-12-31'),
    'wf_2022': ('2022-01-03', '2022-12-30'),
    'wf_2023_2024': ('2023-01-03', '2024-12-31'),
}

BULL_GUARD = BullParticipationGuardConfig(enabled=True)
EQUITY_GUARD = EquityDrawdownControlConfig(
    enabled=True,
    start_drawdown=0.12,
    full_drawdown=0.22,
    min_target=0.25,
    sell_trade_threshold=0.0,
    sell_trade_lock_days=0,
    max_anchor_gap_180=0.0,
    min_below_anchor_ratio_63=0.50,
    min_vix_percentile_252=0.60,
    max_mom_21=0.05,
)
DEEP_LABEL = {
    'max_mom_5': 0.0495,
    'max_drawdown_63': -0.5875,
    'max_base_target': 0.25,
}


def finite(value: Any) -> Any:
    if isinstance(value, (np.integer, np.floating)):
        value = value.item()
    if isinstance(value, float) and not np.isfinite(value):
        return None
    return value


def clean_json(obj: Any) -> Any:
    if isinstance(obj, dict):
        return {str(key): clean_json(value) for key, value in obj.items()}
    if isinstance(obj, list):
        return [clean_json(value) for value in obj]
    return finite(obj)


def run_counterfactual(
    slow: pd.DataFrame,
    fast: pd.DataFrame,
    *,
    label: str,
    trade_threshold: float,
    trade_lock_days: int,
    bull_guard: bool,
    equity_guard: bool,
    deep_guard: bool,
    min_target: float | None = None,
    max_target: float | None = None,
):
    deep_cfg = DeepCapitulationReentryGuardConfig(
        enabled=deep_guard,
        min_target=float(min_target or 0.30),
        max_target=float(max_target or 0.45),
        max_base_target=float(DEEP_LABEL['max_base_target']),
        max_mom_5=float(DEEP_LABEL['max_mom_5']),
        max_drawdown_63=float(DEEP_LABEL['max_drawdown_63']),
    )
    return build_budget_usage_counterfactual(
        slow,
        fast,
        price,
        slow_label=SLOW_PROFILE,
        fast_label=FAST_PROFILE,
        composed_label=label,
        description='fixed-actor deep-capitulation reentry diagnostic',
        profile=label,
        config=TwoSpeedBudgetConfig(trade_threshold=float(trade_threshold), trade_lock_days=int(trade_lock_days)),
        bull_guard=BULL_GUARD if bull_guard else BullParticipationGuardConfig(enabled=False),
        feature_frame=features if equity_guard or deep_guard else None,
        deep_capitulation_guard=deep_cfg,
        drawdown_control=EQUITY_GUARD if equity_guard else EquityDrawdownControlConfig(enabled=False),
        windows=WINDOWS,
    )


def flatten_metrics(payload: dict, label: str) -> dict[str, Any]:
    variant = payload['variants'][label]
    out: dict[str, Any] = {}
    for prefix, metrics in [('current', variant['test_proxy'])]:
        for key, value in metrics.items():
            out[f'{prefix}_{key}'] = value
    for window, metrics in variant.get('slices', {}).items():
        for key, value in metrics.items():
            out[f'{window}_{key}'] = value
    return out


def changed_days(candidate: pd.DataFrame, baseline: pd.DataFrame) -> int:
    joined = candidate[['datetime', 'position']].merge(
        baseline[['datetime', 'position']], on='datetime', how='inner', suffixes=('_candidate', '_baseline')
    )
    return int((joined['position_candidate'].sub(joined['position_baseline']).abs() > 1e-6).sum())


def score_spec(slow: pd.DataFrame, fast: pd.DataFrame, spec: dict[str, Any], *, baseline_daily: pd.DataFrame | None = None) -> tuple[dict[str, Any], pd.DataFrame, dict]:
    result = run_counterfactual(slow, fast, **spec)
    label = spec['label']
    row = {'label': label, **{k: v for k, v in spec.items() if k != 'label'}, **flatten_metrics(result.payload, label)}
    row['deep_days'] = int(result.daily.get('deep_capitulation_reentry_guard', pd.Series(False, index=result.daily.index)).sum())
    row['equity_control_days'] = int(result.daily.get('equity_drawdown_control', pd.Series(False, index=result.daily.index)).sum())
    if baseline_daily is not None:
        row['changed_days_vs_baseline'] = changed_days(result.daily, baseline_daily)
    row['strict_goal_gate'] = bool(row['current_relative_return'] >= 30.0 and row['current_max_dd'] <= 15.0)
    row['diagnostic_excess_gate'] = bool(row['current_relative_return'] >= 30.0)
    return row, result.daily, result.payload


In [4]:
control_specs = [
    {
        'label': 'base_two_speed_thr002_lock3',
        'trade_threshold': 0.02,
        'trade_lock_days': 3,
        'bull_guard': False,
        'equity_guard': False,
        'deep_guard': False,
    },
    {
        'label': 'bull_guard_control_thr002_lock3',
        'trade_threshold': 0.02,
        'trade_lock_days': 3,
        'bull_guard': True,
        'equity_guard': False,
        'deep_guard': False,
    },
    {
        'label': 'equity_guard_control_thr002_lock3',
        'trade_threshold': 0.02,
        'trade_lock_days': 3,
        'bull_guard': True,
        'equity_guard': True,
        'deep_guard': False,
    },
]

control_rows: list[dict[str, Any]] = []
control_daily: dict[str, pd.DataFrame] = {}
control_payloads: dict[str, dict] = {}
for spec in control_specs:
    row, daily, payload = score_spec(slow_current, fast_current, spec)
    row['kind'] = 'control'
    control_rows.append(row)
    control_daily[spec['label']] = daily
    control_payloads[spec['label']] = payload

baseline_for_change = control_daily['equity_guard_control_thr002_lock3']
equity_control_metric = next(row for row in control_rows if row['label'] == 'equity_guard_control_thr002_lock3')

candidate_rows: list[dict[str, Any]] = []
candidate_daily: dict[str, pd.DataFrame] = {}
candidate_payloads: dict[str, dict] = {}
for min_target in [0.25, 0.30, 0.35, 0.40]:
    for max_target in [0.35, 0.45, 0.55]:
        if max_target < min_target:
            continue
        for trade_threshold in [0.00, 0.02, 0.04]:
            for trade_lock_days in [0, 1, 3, 5]:
                label = f'deep_cap_m{min_target:.2f}_x{max_target:.2f}_thr{trade_threshold:.2f}_lock{trade_lock_days}'.replace('.', 'd')
                spec = {
                    'label': label,
                    'trade_threshold': trade_threshold,
                    'trade_lock_days': trade_lock_days,
                    'bull_guard': True,
                    'equity_guard': True,
                    'deep_guard': True,
                    'min_target': min_target,
                    'max_target': max_target,
                }
                row, daily, payload = score_spec(slow_current, fast_current, spec, baseline_daily=baseline_for_change)
                row['kind'] = 'candidate'
                row['maxdd_improvement_vs_equity_control'] = equity_control_metric['current_max_dd'] - row['current_max_dd']
                row['return_delta_vs_equity_control'] = row['current_return'] - equity_control_metric['current_return']
                row['diagnostic_gate'] = bool(
                    row['current_relative_return'] >= 30.0
                    and row['current_max_dd'] <= equity_control_metric['current_max_dd']
                    and row['current_return'] >= equity_control_metric['current_return']
                    and row['changed_days_vs_baseline'] > 0
                )
                row['objective_loss'] = (
                    max(row['current_max_dd'] - 15.0, 0.0) * 10.0
                    + max(30.0 - row['current_relative_return'], 0.0) * 5.0
                    - min(row['current_relative_return'], 3000.0) / 1000.0
                    - max(row['maxdd_improvement_vs_equity_control'], 0.0)
                )
                candidate_rows.append(row)
                candidate_daily[label] = daily
                candidate_payloads[label] = payload

current_scores = pd.DataFrame([*control_rows, *candidate_rows]).sort_values(
    ['kind', 'diagnostic_gate', 'objective_loss', 'current_max_dd'],
    ascending=[True, False, True, True],
    na_position='last',
).reset_index(drop=True)
current_scores.to_csv(OUT_DIR / 'current_candidate_scores.csv', index=False)

cols = [
    'kind', 'label', 'current_return', 'current_relative_return', 'current_max_dd', 'current_sharpe', 'current_avg_pos',
    'test_2022_return', 'test_2022_max_dd', 'test_2023_2024_return', 'test_2023_2024_max_dd',
    'deep_days', 'equity_control_days', 'changed_days_vs_baseline', 'diagnostic_gate', 'strict_goal_gate', 'objective_loss',
]
current_scores[[col for col in cols if col in current_scores.columns]].head(12)


,kind,label,current_return,current_relative_return,current_max_dd,current_sharpe,current_avg_pos,test_2022_return,test_2022_max_dd,test_2023_2024_return,test_2023_2024_max_dd,deep_days,equity_control_days,changed_days_vs_baseline,diagnostic_gate,strict_goal_gate,objective_loss
0,candidate,deep_cap_m0d30_x0d35_thr0d02_lock3,3718.597739,2902.085239,32.809415,1.394646,0.457935,-7.798904,22.956223,242.611397,32.019815,28,104,134.0,True,False,174.193563
1,candidate,deep_cap_m0d30_x0d45_thr0d02_lock3,3718.597739,2902.085239,32.809415,1.394646,0.457935,-7.798904,22.956223,242.611397,32.019815,28,104,134.0,True,False,174.193563
2,candidate,deep_cap_m0d30_x0d55_thr0d02_lock3,3718.597739,2902.085239,32.809415,1.394646,0.457935,-7.798904,22.956223,242.611397,32.019815,28,104,134.0,True,False,174.193563
3,candidate,deep_cap_m0d35_x0d35_thr0d02_lock3,3723.318627,2906.806127,33.627039,1.392513,0.458915,-6.547721,22.956223,246.170859,32.019815,28,100,134.0,True,False,183.182709
4,candidate,deep_cap_m0d35_x0d45_thr0d02_lock3,3723.318627,2906.806127,33.627039,1.392513,0.458915,-6.547721,22.956223,246.170859,32.019815,28,100,134.0,True,False,183.182709
5,candidate,deep_cap_m0d35_x0d55_thr0d02_lock3,3723.318627,2906.806127,33.627039,1.392513,0.458915,-6.547721,22.956223,246.170859,32.019815,28,100,134.0,True,False,183.182709
6,candidate,deep_cap_m0d40_x0d45_thr0d04_lock3,3530.968378,2714.455879,33.655625,1.368641,0.462316,-9.776734,25.223282,267.683963,30.577547,28,60,561.0,True,False,183.689503
7,candidate,deep_cap_m0d40_x0d55_thr0d04_lock3,3530.968378,2714.455879,33.655625,1.368641,0.462316,-9.776734,25.223282,267.683963,30.577547,28,60,561.0,True,False,183.689503
8,candidate,deep_cap_m0d35_x0d35_thr0d04_lock3,3393.367741,2576.855241,33.655625,1.358310,0.461537,-10.963758,25.223282,258.466231,30.577547,28,60,561.0,True,False,183.827104
9,candidate,deep_cap_m0d35_x0d45_thr0d04_lock3,3393.367741,2576.855241,33.655625,1.358310,0.461537,-10.963758,25.223282,258.466231,30.577547,28,60,561.0,True,False,183.827104


In [5]:
def ensure_wf_export(profile: str, tag: str, start: str, end: str) -> Path:
    output = INPUT_DIR / f'{profile}_{tag}_soxl_daily.csv'
    if output.exists():
        return output
    cmd = [
        sys.executable,
        str(REPO_ROOT / 'scripts' / 'export_profile_daily_backtest.py'),
        '--profile', profile,
        '--tag', tag,
        '--output', str(output),
        '--test-start', start,
        '--test-end', end,
    ]
    subprocess.run(cmd, cwd=REPO_ROOT, check=True)
    return output

wf_inputs: dict[str, dict[str, str]] = {}
for tag, (start, end) in WF_WINDOWS.items():
    slow_path = ensure_wf_export(SLOW_PROFILE, tag, start, end)
    fast_path = ensure_wf_export(FAST_PROFILE, tag, start, end)
    wf_inputs[tag] = {
        'start': start,
        'end': end,
        'slow_path': str(slow_path.relative_to(REPO_ROOT)),
        'fast_path': str(fast_path.relative_to(REPO_ROOT)),
    }

manifest = json.loads((OUT_DIR / 'input_manifest.json').read_text(encoding='utf-8'))
manifest['walkforward_inputs'] = wf_inputs
(OUT_DIR / 'input_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
wf_inputs


{'wf_2020': {'start': '2020-01-02',
  'end': '2020-12-31',
  'slow_path': 'runtime/execution/card/deep_capitulation_reentry_guard_v1/inputs/balanced_rule_residual_v19_reg_wf_2020_soxl_daily.csv',
  'fast_path': 'runtime/execution/card/deep_capitulation_reentry_guard_v1/inputs/balanced_rule_residual_v22_wf_2020_soxl_daily.csv'},
 'wf_2022': {'start': '2022-01-03',
  'end': '2022-12-30',
  'slow_path': 'runtime/execution/card/deep_capitulation_reentry_guard_v1/inputs/balanced_rule_residual_v19_reg_wf_2022_soxl_daily.csv',
  'fast_path': 'runtime/execution/card/deep_capitulation_reentry_guard_v1/inputs/balanced_rule_residual_v22_wf_2022_soxl_daily.csv'},
 'wf_2023_2024': {'start': '2023-01-03',
  'end': '2024-12-31',
  'slow_path': 'runtime/execution/card/deep_capitulation_reentry_guard_v1/inputs/balanced_rule_residual_v19_reg_wf_2023_2024_soxl_daily.csv',
  'fast_path': 'runtime/execution/card/deep_capitulation_reentry_guard_v1/inputs/balanced_rule_residual_v22_wf_2023_2024_soxl_daily.cs

In [6]:
eligible_current = current_scores[current_scores['kind'].eq('candidate') & current_scores['diagnostic_excess_gate']].copy()
if eligible_current.empty:
    top_labels = current_scores[current_scores['kind'].eq('candidate')].head(8)['label'].tolist()
else:
    top_labels = eligible_current.sort_values(['diagnostic_gate', 'objective_loss', 'current_max_dd'], ascending=[False, True, True]).head(8)['label'].tolist()

spec_by_label = {row['label']: row for row in candidate_rows}
control_by_label = {row['label']: row for row in control_rows}

wf_rows: list[dict[str, Any]] = []
for tag, paths in wf_inputs.items():
    slow_wf = read_daily(REPO_ROOT / paths['slow_path'])
    fast_wf = read_daily(REPO_ROOT / paths['fast_path'])
    wf_start, wf_end = paths['start'], paths['end']
    wf_price = price.loc[wf_start:wf_end]

    # Controls are scored in their own slice with the global price/features frame; metrics are then recomputed on the WF price range.
    for control_label, spec in [(item['label'], item) for item in control_specs]:
        row, daily, payload = score_spec(slow_wf, fast_wf, spec)
        metric = compute_metrics(daily, wf_price)
        wf_rows.append({
            'kind': 'control',
            'wf_window': tag,
            'label': control_label,
            **{f'wf_{key}': value for key, value in metric.items()},
            'deep_days': int(daily.get('deep_capitulation_reentry_guard', pd.Series(False, index=daily.index)).sum()),
            'equity_control_days': int(daily.get('equity_drawdown_control', pd.Series(False, index=daily.index)).sum()),
            'changed_days_vs_equity_control': 0,
        })

    eq_daily = run_counterfactual(slow_wf, fast_wf, **control_specs[2]).daily
    for label in top_labels:
        base = spec_by_label[label]
        spec = {
            'label': label,
            'trade_threshold': float(base['trade_threshold']),
            'trade_lock_days': int(base['trade_lock_days']),
            'bull_guard': True,
            'equity_guard': True,
            'deep_guard': True,
            'min_target': float(base['min_target']),
            'max_target': float(base['max_target']),
        }
        result = run_counterfactual(slow_wf, fast_wf, **spec)
        metric = compute_metrics(result.daily, wf_price)
        wf_rows.append({
            'kind': 'candidate',
            'wf_window': tag,
            'label': label,
            'min_target': spec['min_target'],
            'max_target': spec['max_target'],
            'trade_threshold': spec['trade_threshold'],
            'trade_lock_days': spec['trade_lock_days'],
            **{f'wf_{key}': value for key, value in metric.items()},
            'deep_days': int(result.daily.get('deep_capitulation_reentry_guard', pd.Series(False, index=result.daily.index)).sum()),
            'equity_control_days': int(result.daily.get('equity_drawdown_control', pd.Series(False, index=result.daily.index)).sum()),
            'changed_days_vs_equity_control': changed_days(result.daily, eq_daily),
        })

walkforward_scores = pd.DataFrame(wf_rows)
walkforward_scores.to_csv(OUT_DIR / 'walkforward_candidate_scores.csv', index=False)
walkforward_scores.sort_values(['wf_window', 'kind', 'wf_max_dd', 'wf_relative_return']).head(20)


,kind,wf_window,label,wf_rows,wf_start,wf_end,wf_return,wf_sharpe,wf_max_dd,wf_avg_pos,...,wf_buy_and_hold_return,wf_relative_return,wf_trades,deep_days,equity_control_days,changed_days_vs_equity_control,min_target,max_target,trade_threshold,trade_lock_days
9,candidate,wf_2020,deep_cap_m0d40_x0d45_thr0d04_lock3,252,2020-01-02,2020-12-30,88.447559,1.306323,25.907428,0.418427,...,58.093571,30.353988,31,6,2,97,0.40,0.45,0.04,3.0
10,candidate,wf_2020,deep_cap_m0d40_x0d55_thr0d04_lock3,252,2020-01-02,2020-12-30,88.447559,1.306323,25.907428,0.418427,...,58.093571,30.353988,31,6,2,97,0.40,0.55,0.04,3.0
6,candidate,wf_2020,deep_cap_m0d35_x0d35_thr0d02_lock3,252,2020-01-02,2020-12-30,101.400186,1.407117,26.256455,0.421466,...,58.093571,43.306616,34,6,2,5,0.35,0.35,0.02,3.0
7,candidate,wf_2020,deep_cap_m0d35_x0d45_thr0d02_lock3,252,2020-01-02,2020-12-30,101.400186,1.407117,26.256455,0.421466,...,58.093571,43.306616,34,6,2,5,0.35,0.45,0.02,3.0
8,candidate,wf_2020,deep_cap_m0d35_x0d55_thr0d02_lock3,252,2020-01-02,2020-12-30,101.400186,1.407117,26.256455,0.421466,...,58.093571,43.306616,34,6,2,5,0.35,0.55,0.02,3.0
3,candidate,wf_2020,deep_cap_m0d30_x0d35_thr0d02_lock3,252,2020-01-02,2020-12-30,96.588263,1.381838,26.256455,0.419786,...,58.093571,38.494692,35,6,2,13,0.30,0.35,0.02,3.0
4,candidate,wf_2020,deep_cap_m0d30_x0d45_thr0d02_lock3,252,2020-01-02,2020-12-30,96.588263,1.381838,26.256455,0.419786,...,58.093571,38.494692,35,6,2,13,0.30,0.45,0.02,3.0
5,candidate,wf_2020,deep_cap_m0d30_x0d55_thr0d02_lock3,252,2020-01-02,2020-12-30,96.588263,1.381838,26.256455,0.419786,...,58.093571,38.494692,35,6,2,13,0.30,0.55,0.02,3.0
0,control,wf_2020,base_two_speed_thr002_lock3,252,2020-01-02,2020-12-30,81.423588,1.252922,25.907428,0.406122,...,58.093571,23.330017,31,0,0,0,NaN,NaN,NaN,NaN
1,control,wf_2020,bull_guard_control_thr002_lock3,252,2020-01-02,2020-12-30,111.070844,1.487506,26.256455,0.417535,...,58.093571,52.977274,33,0,0,0,NaN,NaN,NaN,NaN


In [7]:
wf_pivot = walkforward_scores[walkforward_scores['kind'].eq('candidate')].pivot_table(
    index='label',
    columns='wf_window',
    values=['wf_return', 'wf_relative_return', 'wf_max_dd', 'changed_days_vs_equity_control', 'deep_days'],
    aggfunc='first',
)
wf_pivot.columns = ['__'.join(col).strip() for col in wf_pivot.columns.to_flat_index()]
wf_pivot = wf_pivot.reset_index()

candidate_summary = current_scores[current_scores['kind'].eq('candidate')].merge(wf_pivot, on='label', how='left')
for col in ['wf_relative_return__wf_2022', 'wf_return__wf_2022', 'wf_max_dd__wf_2022', 'wf_return__wf_2023_2024', 'wf_max_dd__wf_2023_2024']:
    if col not in candidate_summary.columns:
        candidate_summary[col] = np.nan

candidate_summary['wf_penalty'] = (
    candidate_summary['wf_max_dd__wf_2022'].fillna(100.0).clip(lower=15.0).sub(15.0)
    + candidate_summary['wf_max_dd__wf_2023_2024'].fillna(100.0).clip(lower=15.0).sub(15.0)
    + np.maximum(0.0, -candidate_summary['wf_relative_return__wf_2022'].fillna(-1000.0)) / 10.0
)
candidate_summary['selection_loss'] = candidate_summary['objective_loss'] + candidate_summary['wf_penalty']

selected_row = candidate_summary.sort_values(
    ['diagnostic_gate', 'selection_loss', 'current_max_dd'],
    ascending=[False, True, True],
    na_position='last',
).iloc[0]
selected_label = str(selected_row['label'])
selected_daily = candidate_daily[selected_label]
selected_payload = candidate_payloads[selected_label]

dash = dashboard_daily_frame(selected_daily, price)
dash.to_csv(OUT_DIR / 'selected_variant_daily.csv', index=False)


def rolling_poor_windows(daily: pd.DataFrame, price_series: pd.Series, *, window: int = 63, top_n: int = 20) -> pd.DataFrame:
    scored = align_price(daily[['datetime', 'position']], price_series)
    scored = scored.sort_values('datetime').reset_index(drop=True)
    prices = scored['price'].to_numpy(dtype=float)
    positions = scored['position'].to_numpy(dtype=float)
    returns = prices[1:] / prices[:-1] - 1.0
    strat_returns = positions[:-1] * returns
    strategy_nav = np.concatenate([[1.0], np.cumprod(1.0 + strat_returns)])
    buyhold_nav = np.concatenate([[1.0], np.cumprod(1.0 + returns)])
    rows: list[dict[str, Any]] = []
    for start_idx in range(0, len(scored) - window):
        end_idx = start_idx + window
        strat_slice = strategy_nav[start_idx:end_idx + 1]
        bh_slice = buyhold_nav[start_idx:end_idx + 1]
        peak = np.maximum.accumulate(strat_slice)
        max_dd = abs(float(np.min(strat_slice / np.maximum(peak, 1e-12) - 1.0)) * 100.0)
        strategy_return = float(strat_slice[-1] / strat_slice[0] - 1.0) * 100.0
        buyhold_return = float(bh_slice[-1] / bh_slice[0] - 1.0) * 100.0
        rows.append({
            'window_start': scored['datetime'].iloc[start_idx].date().isoformat(),
            'window_end': scored['datetime'].iloc[end_idx].date().isoformat(),
            'strategy_return': strategy_return,
            'buyhold_return': buyhold_return,
            'excess_pp': strategy_return - buyhold_return,
            'strategy_max_dd': max_dd,
            'avg_position': float(np.mean(positions[start_idx:end_idx + 1])),
        })
    return pd.DataFrame(rows).sort_values(['excess_pp', 'strategy_max_dd'], ascending=[True, False]).head(top_n).reset_index(drop=True)

poor_windows = rolling_poor_windows(selected_daily, price, window=63, top_n=20)
poor_windows.to_csv(OUT_DIR / 'poor_windows.csv', index=False)

strict_goal_gate = bool(selected_row['current_relative_return'] >= 30.0 and selected_row['current_max_dd'] <= 15.0)
wf_candidate = walkforward_scores[walkforward_scores['label'].eq(selected_label)].copy()
wf_goal_gate = bool((wf_candidate['wf_relative_return'] >= 30.0).all() and (wf_candidate['wf_max_dd'] <= 15.0).all()) if not wf_candidate.empty else False
promote_gate = bool(strict_goal_gate and wf_goal_gate)

selection = {
    'experiment': EXPERIMENT,
    'selected_label': selected_label,
    'selected_spec': {
        'min_target': finite(selected_row.get('min_target')),
        'max_target': finite(selected_row.get('max_target')),
        'trade_threshold': finite(selected_row.get('trade_threshold')),
        'trade_lock_days': int(selected_row.get('trade_lock_days')),
        **DEEP_LABEL,
    },
    'strict_goal_gate': strict_goal_gate,
    'wf_goal_gate': wf_goal_gate,
    'promote_gate': promote_gate,
    'decision': 'diagnostic_keep_no_promotion' if not promote_gate else 'promotion_candidate_requires_manual_review',
    'reason': 'MaxDD remains above the 15% hard objective' if not strict_goal_gate else 'strict OOS passed; walk-forward gate checked separately',
    'current_metrics': {key: finite(selected_row[key]) for key in selected_row.index if str(key).startswith('current_')},
    'walkforward_metrics': clean_json(wf_candidate.to_dict(orient='records')),
    'artifacts': {
        'current_candidate_scores': str((OUT_DIR / 'current_candidate_scores.csv').relative_to(REPO_ROOT)),
        'walkforward_candidate_scores': str((OUT_DIR / 'walkforward_candidate_scores.csv').relative_to(REPO_ROOT)),
        'selected_variant_daily': str((OUT_DIR / 'selected_variant_daily.csv').relative_to(REPO_ROOT)),
        'poor_windows': str((OUT_DIR / 'poor_windows.csv').relative_to(REPO_ROOT)),
    },
}
(OUT_DIR / 'selection.json').write_text(json.dumps(clean_json(selection), indent=2), encoding='utf-8')

wf_readout = []
wf_risk_flags: list[str] = []
for _, row in wf_candidate.sort_values('wf_window').iterrows():
    wf_readout.append({
        'wf_window': row['wf_window'],
        'return': finite(row['wf_return']),
        'relative_return': finite(row['wf_relative_return']),
        'max_dd': finite(row['wf_max_dd']),
        'deep_days': int(row['deep_days']),
        'changed_days_vs_equity_control': int(row['changed_days_vs_equity_control']),
    })
    eq_match = walkforward_scores[
        walkforward_scores['wf_window'].eq(row['wf_window'])
        & walkforward_scores['label'].eq('equity_guard_control_thr002_lock3')
    ]
    if not eq_match.empty:
        eq = eq_match.iloc[0]
        if row['wf_return'] < eq['wf_return']:
            wf_risk_flags.append(f"{row['wf_window']}: return {row['wf_return'] - eq['wf_return']:.2f}pp versus equity-control")
        if row['wf_max_dd'] > eq['wf_max_dd']:
            wf_risk_flags.append(f"{row['wf_window']}: MaxDD {row['wf_max_dd'] - eq['wf_max_dd']:.2f}pp versus equity-control")
        if int(row['changed_days_vs_equity_control']) == 0:
            wf_risk_flags.append(f"{row['wf_window']}: no order-generation change versus equity-control")

wf_table_md = '\n'.join(
    f"- `{item['wf_window']}`: return `{item['return']:.2f}%`, excess `{item['relative_return']:.2f}pp`, MaxDD `{item['max_dd']:.2f}%`, changed days `{item['changed_days_vs_equity_control']}`"
    for item in wf_readout
)
risk_flag_text = '; '.join(wf_risk_flags) if wf_risk_flags else 'none versus the equity-control comparator'
worst_poor_window = clean_json(poor_windows.iloc[0].to_dict()) if not poor_windows.empty else {}

summary = {
    'experiment': EXPERIMENT,
    'baseline': 'v19_reg slow actor + v22 fast usage actor with bull/equity controls',
    'decision': selection['decision'],
    'promote_gate': promote_gate,
    'selected_label': selected_label,
    'selected_current_return': finite(selected_row['current_return']),
    'selected_current_relative_return': finite(selected_row['current_relative_return']),
    'selected_current_max_dd': finite(selected_row['current_max_dd']),
    'selected_current_sharpe': finite(selected_row['current_sharpe']),
    'selected_deep_days': int(selected_row['deep_days']),
    'selected_changed_days_vs_equity_control': int(selected_row['changed_days_vs_baseline']),
    'equity_control_current_return': finite(equity_control_metric['current_return']),
    'equity_control_current_relative_return': finite(equity_control_metric['current_relative_return']),
    'equity_control_current_max_dd': finite(equity_control_metric['current_max_dd']),
    'selected_walkforward': wf_readout,
    'walkforward_risk_flags': wf_risk_flags,
    'worst_poor_window': worst_poor_window,
    'walkforward_rows': int(len(walkforward_scores)),
    'next_action': 'Do not promote or retrain PPO from this alone; use the notebook evidence to design explicit downside-hedge or regime-aware risk control if the 15% MaxDD target remains mandatory.',
}
(OUT_DIR / 'summary.json').write_text(json.dumps(clean_json(summary), indent=2), encoding='utf-8')

conclusion = f"""
## Runtime conclusion

Selected variant: `{selected_label}`.

Strict OOS readout: return `{selected_row['current_return']:.2f}%`, excess versus buy-and-hold `{selected_row['current_relative_return']:.2f}pp`, Sharpe `{selected_row['current_sharpe']:.3f}`, MaxDD `{selected_row['current_max_dd']:.2f}%`, deep-cap guard days `{int(selected_row['deep_days'])}`, changed days versus the equity-control baseline `{int(selected_row['changed_days_vs_baseline'])}`.

Walk-forward readout:
{wf_table_md}

Risk flags versus the equity-control comparator: {risk_flag_text}.

Worst 63-trading-day poor window: `{worst_poor_window.get('window_start')}` to `{worst_poor_window.get('window_end')}`, strategy `{worst_poor_window.get('strategy_return'):.2f}%`, buy-and-hold `{worst_poor_window.get('buyhold_return'):.2f}%`, excess `{worst_poor_window.get('excess_pp'):.2f}pp`.

The guard is a useful diagnostic: it raises low targets on sparse deep-capitulation dates and improves strict-OOS return versus the equity-control line while producing actual order-generation changes. It is **not** a promotion candidate because MaxDD remains above the hard `15%` target and walk-forward checks still show regression/no-change risk. The next loop should not be another target-floor threshold tweak; if the `15%` MaxDD target is mandatory, the missing layer is explicit downside protection or a stronger regime-aware risk controller.
"""
display(Markdown(conclusion))

pd.DataFrame([summary])



## Runtime conclusion

Selected variant: `deep_cap_m0d30_x0d35_thr0d02_lock3`.

Strict OOS readout: return `3718.60%`, excess versus buy-and-hold `2902.09pp`, Sharpe `1.395`, MaxDD `32.81%`, deep-cap guard days `28`, changed days versus the equity-control baseline `134`.

Walk-forward readout:
- `wf_2020`: return `96.59%`, excess `38.49pp`, MaxDD `26.26%`, changed days `13`
- `wf_2022`: return `-32.99%`, excess `53.59pp`, MaxDD `36.75%`, changed days `80`
- `wf_2023_2024`: return `235.11%`, excess `36.61pp`, MaxDD `33.78%`, changed days `0`

Risk flags versus the equity-control comparator: wf_2020: return -13.25pp versus equity-control; wf_2022: return -1.98pp versus equity-control; wf_2022: MaxDD 4.31pp versus equity-control; wf_2023_2024: no order-generation change versus equity-control.

Worst 63-trading-day poor window: `2026-02-09` to `2026-05-11`, strategy `2.78%`, buy-and-hold `197.39%`, excess `-194.61pp`.

The guard is a useful diagnostic: it raises low targets on sparse deep-capitulation dates and improves strict-OOS return versus the equity-control line while producing actual order-generation changes. It is **not** a promotion candidate because MaxDD remains above the hard `15%` target and walk-forward checks still show regression/no-change risk. The next loop should not be another target-floor threshold tweak; if the `15%` MaxDD target is mandatory, the missing layer is explicit downside protection or a stronger regime-aware risk controller.


,experiment,baseline,decision,promote_gate,selected_label,selected_current_return,selected_current_relative_return,selected_current_max_dd,selected_current_sharpe,selected_deep_days,selected_changed_days_vs_equity_control,equity_control_current_return,equity_control_current_relative_return,equity_control_current_max_dd,selected_walkforward,walkforward_risk_flags,worst_poor_window,walkforward_rows,next_action
0,deep_capitulation_reentry_guard_v1,v19_reg slow actor + v22 fast usage actor with...,diagnostic_keep_no_promotion,False,deep_cap_m0d30_x0d35_thr0d02_lock3,3718.597739,2902.085239,32.809415,1.394646,28,134,2899.799942,2083.287443,33.807919,"[{'wf_window': 'wf_2020', 'return': 96.5882630...",[wf_2020: return -13.25pp versus equity-contro...,"{'window_start': '2026-02-09', 'window_end': '...",33,Do not promote or retrain PPO from this alone;...


## Sidecar artifacts

This notebook is the primary human-readable runtime conclusion. Sidecars are machine-readable evidence for indexing and follow-up:

- `current_candidate_scores.csv`
- `walkforward_candidate_scores.csv`
- `selected_variant_daily.csv`
- `poor_windows.csv`
- `input_manifest.json`
- `selection.json`
- `summary.json`
